## 5. Ensemble learning - Stacking method
In this part, ensemble learning was used, with stacking as the method for obtaining the final prediction. First, three bottom-layer models were trained: Logistic Regression, Random Fores and XGBoost. This time, only one group of features was selected as variables, chosen using the Boruta method. Then, various voting methods (Average Voting, Weighted Average Voting, Majority Voting, Stacking) were used to select the final prediction, and the method that achieved the best result on the validation set was selected as the final model.

At the end of the script, the results of all three models/methods used in the project (Random Forrest, XGBoost, Stacking) will be compared, and the best method will be selected based on the result obtained on the test set.

In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import RFECV
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from ReliefF import ReliefF
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import numpy as np

import pickle

from sklearn.model_selection import train_test_split

import xgboost as xgb

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 150)

In [2]:
df = pd.read_csv('C:\\Users\\kubas\\OneDrive\\Documents\\ML2\\ProjectClassification\\train_fe.csv')

fr = pd.read_excel('C:\\Users\\kubas\\OneDrive\\Documents\\ML2\\ProjectClassification\\feature_ranking.xlsx', index_col=0)


### Spliting train dataset into train and validation sets

In [3]:
X = df.drop(columns=['claim_status'])
y = df['claim_status']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [4]:
df_res = pd.concat([pd.DataFrame(X_train, columns=X_train.columns),
                                pd.Series(y_train, name='claim_status')],
                                axis=1)

In [5]:
df_val = pd.concat([X_val, y_val], axis=1)

#### Features group:
Only one group of features was selected, I decided to choose features chosen using Boruta method because this features group achieved the best results in previous models.

In [14]:
br_features = ['subscription_length_sqrt',
 'subscription_length',
 'ncap_rating',
 'vehicle_age',
 'region_density_sqrt',
 'gross_weight',
 'displacement',
 'width',
 'region_density',
 'customer_age',
 'customer_age_log',
 'vehicle_age_cut']

### Ensembling 3 models - Logistic Regression, Random Forrest, XGBoost
This function is used to train a model on a training set using cross-validation, and then returns the balanced accuracy metric calculated on the validation set. It also calculates the mean and standard deviation of the balanced accuracy metric on the training set calculated during cross-validation

In [17]:
def perform_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    cv=5,
    random_state=42
):
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state
    )

    cv_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model.fit(X_tr, y_tr)

        preds = model.predict(X_va)
        score = balanced_accuracy_score(y_va, preds)

        cv_scores.append(score)

    # Refit on full training data
    model.fit(X_train, y_train)

    # Validation performance
    val_preds = model.predict(X_val)
    val_score = balanced_accuracy_score(y_val, val_preds)

    return {
        "cv_balanced_accuracy_mean": np.mean(cv_scores),
        "cv_balanced_accuracy_std": np.std(cv_scores),
        "val_balanced_accuracy": val_score
    }

##### The following three bottom-layer models were selected: Logistic Regression, Random Forests, and XGBoost. For the latter two models, parameters for the br_features group obtained earlier during hyperparameter tuning for these models were used.

In [20]:
models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        n_jobs=-1
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=10,
        max_features='sqrt',
        max_samples=0.8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight= 15,
        n_estimators=600,
        max_depth=4,
        gamma = 0.25,
        learning_rate = 0.01,
        subsample=0.8,
        reg_lambda = 10,
        random_state=42,
        n_jobs=-1
    )
}

In [29]:
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    results[name] = perform_model(
        model,
        df_res.loc[:, br_features],
        y_train,
        df_val.loc[:, br_features],
        y_val
    )


Training LogisticRegression...

Training RandomForest...

Training XGBoost...


In [30]:
results_df = pd.DataFrame(results).T
results_df

,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,val_balanced_accuracy
LogisticRegression,0.571107,0.008988,0.573536
RandomForest,0.611615,0.005341,0.613530
XGBoost,0.609246,0.006972,0.610710


#### Predicting probabilitis for training dataset:

In [32]:
val_preds = {}

for name, model in models.items():
    model.fit(df_res.loc[:, br_features], y_train)
    val_preds[name] = model.predict_proba(
        df_val.loc[:, br_features]
    )[:, 1]

In [33]:
val_preds

{'LogisticRegression': array([0.40953228, 0.63064612, 0.49088275, ..., 0.51701421, 0.33015562,
        0.5626829 ], shape=(9375,)),
 'RandomForest': array([0.30232415, 0.52908554, 0.54732491, ..., 0.44801394, 0.2080294 ,
        0.66265766], shape=(9375,)),
 'XGBoost': array([0.28897852, 0.54520196, 0.5549164 , ..., 0.4542423 , 0.19851944,
        0.6765403 ], shape=(9375,), dtype=float32)}

### Voting

#### Average Voting

In [38]:
ensemble_proba = (
    val_preds["LogisticRegression"] +
    val_preds["RandomForest"] +
    val_preds["XGBoost"]
) / 3

ensemble_pred = (ensemble_proba >= 0.5).astype(int)
balanced_accuracy_score(y_val, ensemble_pred)

0.6042016887745495

#### Weighted Average  Voting

In [39]:
weights = {
    "LogisticRegression": results["LogisticRegression"]["cv_balanced_accuracy_mean"],
    "RandomForest": results["RandomForest"]["cv_balanced_accuracy_mean"],
    "XGBoost": results["XGBoost"]["cv_balanced_accuracy_mean"]
}

total = sum(weights.values())

ensemble_proba = sum(
    weights[name] * val_preds[name]
    for name in val_preds
) / total

ensemble_pred = (ensemble_proba >= 0.5).astype(int)
balanced_accuracy_score(y_val, ensemble_pred)

0.6033155018341942

#### Majority Voting

In [40]:
ensemble_pred = (
    (val_preds["LogisticRegression"] >= 0.5).astype(int) +
    (val_preds["RandomForest"] >= 0.5).astype(int) +
    (val_preds["XGBoost"] >= 0.5).astype(int)
) >= 2

balanced_accuracy_score(y_val, ensemble_pred)

0.6076378254645605

### Stacking

In [41]:
def stacking_classifier(
    base_models,
    meta_model,
    X_train,
    y_train,
    X_val,
    y_val,
    cv=5,
    random_state=42
):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    oof_preds = np.zeros((X_train.shape[0], len(base_models)))

    for i, model in enumerate(base_models):
        for tr_idx, va_idx in skf.split(X_train, y_train):
            X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
            y_tr = y_train.iloc[tr_idx]

            model.fit(X_tr, y_tr)
            oof_preds[va_idx, i] = model.predict_proba(X_va)[:, 1]

    meta_model.fit(oof_preds, y_train)

    val_meta_features = np.column_stack([
        model.fit(X_train, y_train).predict_proba(X_val)[:, 1]
        for model in base_models
    ])

    val_preds = meta_model.predict(val_meta_features)
    val_score = balanced_accuracy_score(y_val, val_preds)

    return val_score

In [59]:
base_models = [
    models["LogisticRegression"],
    models["RandomForest"],
    models["XGBoost"]
]

meta_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

stacking_score = stacking_classifier(
    base_models,
    meta_model,
    df_res.loc[:, br_features],
    y_train,
    df_val.loc[:, br_features],
    y_val
)

print("Stacking balanced accuracy:", stacking_score)

Stacking balanced accuracy: 0.616127638143074


#### Best method of ensembling - stacking

### Testing model on test dataset:

In [71]:
df_test = pd.read_csv('C:\\Users\\kubas\\OneDrive\\Documents\\ML2\\ProjectClassification\\test_fe.csv')

In [72]:
X_test = df_test.loc[:, br_features].values
y_test = df_test.loc[:, "claim_status"].values.ravel()

In [75]:
for model in base_models:
    model.fit(df_res.loc[:, br_features], y_train)

In [77]:
def get_meta_features(models, X):
    return np.column_stack([
        model.predict_proba(X)[:, 1]
        for model in models
    ])
Z_val  = get_meta_features(base_models, df_val.loc[:, br_features])
Z_test = get_meta_features(base_models, X_test)

C:\Users\kubas\miniconda3\envs\cleanenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\kubas\miniconda3\envs\cleanenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [78]:
meta_model.fit(Z_val, y_val)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [79]:
test_preds = meta_model.predict(Z_test)

test_balanced_acc = balanced_accuracy_score(y_test, test_preds)

print(f"Stacking balanced accuracy (TEST): {test_balanced_acc:.4f}")

Stacking balanced accuracy (TEST): 0.6104


### Conclusion
Below, the results of all three models/methods used in the project are compared based on balanced accuracy on the test set. The validation and test sets were the same for all models, so it is possible to compare them. The best result: **balanced accuracy = 0.6109** was achieved by the **XGBoost model** with the following **parameters: {‘gamma’: 0.25, ‘learning_rate’: 0.01, ‘max_depth’: 4, ‘n_estimators’: 600, ‘reg_lambda’: 10}** and features selected using Boruta method.

XGBoost model performs very well with unbalanced data sets by combining strong regularization and flexible loss weighting, which is why it achieved the best results, although the other models also performed well, especially stacking. 

In [1]:
print("Random Forest balanced accuracy on test set: 0.6071")
print("XGBoost balanced accuracy on test set: 0.6109")
print("Stacking balanced accuracy on test set: 0.6104")

Random Forest balanced accuracy on test set: 0.6071
XGBoost balanced accuracy on test set: 0.6109
Stacking balanced accuracy on test set: 0.6104


### Bibliography
* Kaggle, *Insurance Claims Dataset*, https://www.kaggle.com/datasets/litvinenko630/insurance-claims
* Machine Learning 1 and 2: *Course materials*